# CSA 821 Assignment 1: Machine Learning Models for Kenyan Job Posts

**Project:** Next Step (`nextstep.co.ke`)  
**Dataset source:** Reproducible CSV snapshot exported from the live PostgreSQL database (`career_lmi`)  
**Snapshot file:** `notebooks/data/csa821_kenya_seniority_dataset.csv`  
**Export SQL:** `notebooks/sql/csa821_kenya_seniority_dataset_export.sql`  
**Recommended task:** Multiclass classification of job seniority for Kenyan jobs  

**Group member 1:** `Name - Registration Number`  
**Group member 2:** `Name - Registration Number`  
**Group member 3:** `Name - Registration Number`  
**Group member 4:** `Name - Registration Number`  
**Group member 5:** `Name - Registration Number`

## Assignment Fit
- Real-world dataset: Kenyan job-market data exported from the Next Step production dataset.
- Kenyan relevance bonus: this notebook restricts modelling to jobs located in Kenya.
- Preprocessing: missing-value handling, outlier clipping, normalization, one-hot encoding, and feature selection are included.
- Model development: compares Logistic Regression and Linear SVM.
- Optimization: uses `GridSearchCV`.
- Evaluation: reports Accuracy, Precision, Recall, F1-score, ROC-AUC, confusion matrices, and ROC curves.


## Problem Statement

This notebook uses job advertisement text and metadata to predict the `seniority` level of Kenyan job posts.  
The target classes are typically:
- `Entry`
- `Mid-Level`
- `Senior`
- `Executive`

This is a good assignment problem because:
- the target is already present in the production data,
- the class sizes are large enough for supervised learning,
- the features are realistic: job title, job description, source, education, city, quality score, and skill counts.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sqlalchemy import create_engine, text

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectPercentile, chi2
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelBinarizer, MinMaxScaler, OneHotEncoder
from sklearn.svm import LinearSVC

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
dataset_path = project_root / "notebooks" / "data" / "csa821_kenya_seniority_dataset.csv"
sql_path = project_root / "notebooks" / "sql" / "csa821_kenya_seniority_dataset_export.sql"

env_candidates = [
    project_root / ".env",
    Path("/home/nextstep.co.ke/.env"),
]
if load_dotenv is not None:
    for env_path in env_candidates:
        if env_path.exists():
            load_dotenv(env_path, override=False)

DATABASE_URL = os.getenv("DATABASE_URL")

print("Using project root:", project_root)
print("Dataset path:", dataset_path)
print("SQL path:", sql_path)
print("DATABASE_URL available:", bool(DATABASE_URL))


In [ ]:
if dataset_path.exists():
    df = pd.read_csv(dataset_path)
    data_source = f"CSV snapshot: {dataset_path.name}"
else:
    if not DATABASE_URL:
        raise RuntimeError(
            "CSV snapshot not found and DATABASE_URL is not set. Provide the CSV or a PostgreSQL connection."
        )
    engine = create_engine(DATABASE_URL)
    query = text(sql_path.read_text(encoding="utf-8"))
    df = pd.read_sql(query, engine)
    dataset_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(dataset_path, index=False)
    data_source = f"PostgreSQL export regenerated to {dataset_path.name}"

print("Loaded data source:", data_source)
df.head()


In [ ]:
print("Dataset shape:", df.shape)
print("\nClass distribution:")
display(df["seniority"].value_counts().to_frame("count"))

missing_summary = df.isna().sum().sort_values(ascending=False).to_frame("missing_values")
display(missing_summary)

numeric_summary = df[["skill_count", "quality_score", "repost_count"]].describe().T
display(numeric_summary)

df["description_length"] = df["description_clean"].str.len()
for col in ["description_length", "skill_count", "repost_count"]:
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=upper)

df["text"] = (
    df["title_raw"].fillna("")
    + " [SEP] "
    + df["description_clean"].fillna("")
)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
sns.countplot(data=df, y="seniority", order=df["seniority"].value_counts().index, ax=axes[0, 0])
axes[0, 0].set_title("Target Class Distribution")

top_sources = df["source"].value_counts().head(10)
sns.barplot(x=top_sources.values, y=top_sources.index, ax=axes[0, 1])
axes[0, 1].set_title("Top Sources in the Kenya Subset")
axes[0, 1].set_xlabel("Number of jobs")

sns.histplot(df["description_length"], bins=40, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("Description Length Distribution")

top_titles = df["canonical_title"].value_counts().head(12)
sns.barplot(x=top_titles.values, y=top_titles.index, ax=axes[1, 1])
axes[1, 1].set_title("Top Canonical Titles")
axes[1, 1].set_xlabel("Number of jobs")

plt.tight_layout()
plt.show()


## Preprocessing Decisions

The notebook applies the following preprocessing steps:

1. **Missing values**
   - Text fields are filled with empty strings.
   - Categorical fields are imputed with the most frequent value.
   - Numeric fields are imputed with the median.

2. **Outliers**
   - `description_length`, `skill_count`, and `repost_count` are clipped at the 99th percentile.

3. **Normalization / scaling**
   - Numeric features are scaled using `MinMaxScaler`.

4. **Encoding**
   - Categorical variables are transformed using `OneHotEncoder`.
   - Text is transformed using `TfidfVectorizer`.

5. **Feature selection**
   - `SelectPercentile(chi2, ...)` is used to reduce dimensionality and retain the most informative features.


In [ ]:
feature_columns = [
    "text",
    "source",
    "canonical_title",
    "sector",
    "city",
    "region",
    "education",
    "skill_count",
    "quality_score",
    "repost_count",
    "description_length",
]

target_column = "seniority"

X = df[feature_columns].copy()
y = df[target_column].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

text_features = "text"
categorical_features = ["source", "canonical_title", "sector", "city", "region", "education"]
numeric_features = ["skill_count", "quality_score", "repost_count", "description_length"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5, max_features=20000),
            text_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", MinMaxScaler()),
                ]
            ),
            numeric_features,
        ),
    ]
)

log_reg_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("selector", SelectPercentile(score_func=chi2, percentile=35)),
        (
            "model",
            LogisticRegression(max_iter=1500, solver="liblinear", random_state=42),
        ),
    ]
)

svm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("selector", SelectPercentile(score_func=chi2, percentile=35)),
        ("model", LinearSVC(random_state=42)),
    ]
)

log_reg_grid = {
    "selector__percentile": [20, 35, 50],
    "model__C": [0.5, 1.0, 2.0],
}

svm_grid = {
    "selector__percentile": [20, 35, 50],
    "model__C": [0.5, 1.0, 2.0],
}

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])


In [ ]:
log_reg_search = GridSearchCV(
    estimator=log_reg_pipeline,
    param_grid=log_reg_grid,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1,
)

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_grid,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1,
)

log_reg_search.fit(X_train, y_train)
svm_search.fit(X_train, y_train)

print("Best Logistic Regression params:", log_reg_search.best_params_)
print("Best Logistic Regression CV F1:", round(log_reg_search.best_score_, 4))
print()
print("Best Linear SVM params:", svm_search.best_params_)
print("Best Linear SVM CV F1:", round(svm_search.best_score_, 4))


In [ ]:
def evaluate_model(name, estimator, X_test, y_test):
    predictions = estimator.predict(X_test)

    if hasattr(estimator, "predict_proba"):
        scores = estimator.predict_proba(X_test)
    else:
        scores = estimator.decision_function(X_test)

    lb = LabelBinarizer()
    y_test_binary = lb.fit_transform(y_test)
    if y_test_binary.shape[1] == 1:
        y_test_binary = np.hstack([1 - y_test_binary, y_test_binary])

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision_weighted": precision_score(y_test, predictions, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_test, predictions, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_test, predictions, average="weighted", zero_division=0),
        "roc_auc_ovr_weighted": roc_auc_score(y_test_binary, scores, multi_class="ovr", average="weighted"),
    }

    print(f"\n{name} classification report")
    print(classification_report(y_test, predictions, zero_division=0))

    cm = confusion_matrix(y_test, predictions, labels=lb.classes_)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=lb.classes_, yticklabels=lb.classes_)
    plt.title(f"Confusion Matrix: {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    return metrics, lb, scores


results = []
score_store = {}
binarizers = {}

for name, estimator in {
    "Logistic Regression": log_reg_search.best_estimator_,
    "Linear SVM": svm_search.best_estimator_,
}.items():
    metrics, lb, scores = evaluate_model(name, estimator, X_test, y_test)
    results.append(metrics)
    score_store[name] = scores
    binarizers[name] = lb

results_df = pd.DataFrame(results).sort_values("f1_weighted", ascending=False)
display(results_df)


In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_estimator = (
    log_reg_search.best_estimator_
    if best_model_name == "Logistic Regression"
    else svm_search.best_estimator_
)
best_scores = score_store[best_model_name]
lb = binarizers[best_model_name]
y_test_binary = lb.transform(y_test)
if y_test_binary.shape[1] == 1:
    y_test_binary = np.hstack([1 - y_test_binary, y_test_binary])

plt.figure(figsize=(9, 7))
for i, class_name in enumerate(lb.classes_):
    fpr, tpr, _ = roc_curve(y_test_binary[:, i], best_scores[:, i])
    plt.plot(fpr, tpr, label=f"{class_name}")

plt.plot([0, 1], [0, 1], linestyle="--", color="black")
plt.title(f"One-vs-Rest ROC Curves: {best_model_name}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

print("Best model selected:", best_model_name)


## Interpretation and Discussion

Use this section to discuss your final results in your submission:

- Which model performed best on the test set?
- Did hyperparameter tuning improve weighted F1-score?
- Which classes were easier or harder to classify?
- How might data quality issues affect the results?

### Suggested discussion points
- `seniority` is a useful label because it is already operationalized in the dataset.
- The Kenya subset is large and relevant to the assignment bonus criteria.
- Text features are likely the strongest signal because job descriptions often state experience and responsibility level directly.
- Sparse salary data makes regression a weaker assignment choice for this database.
